## Actor Critic Models w/ CartPole
#### Best run with alpha 0.025 and gamma=0.99

![Alt text](best_run_ac.png)


### Reference: Actor-Critic

See the full documentation in the [README](../../README.md#actor-critic).

Actor-Critic methods maintain two separate function approximators:

- **Actor (Policy Network)**: outputs a probability distribution over actions given a state. Parameterized as pi(a|s; theta).
- **Critic (Value Network)**: estimates the value of a state V(s; w). Used to evaluate how good the actor's actions are.

#### How it works

1. The agent selects an action by sampling from the actor's distribution: a ~ pi(a|s; theta)
2. After observing the transition (s, a, r, s'), the critic computes the TD target and advantage:
   - **TD Target**: delta = r + gamma * V(s'; w)
   - **Advantage**: A(s) = delta - V(s; w)
3. The critic is updated to minimize the error between its prediction and the TD target:
   - **Critic Loss**: L_critic = (V(s; w) - (r + gamma * V(s'; w)))^2
4. The actor is updated using the policy gradient, scaled by the advantage:
   - **Policy Loss**: L_actor = -log(pi(a|s; theta)) * A(s)
   - Positive advantage reinforces the action (makes it more likely), negative advantage discourages it

#### Key details
- V(s') must be **detached** from the computation graph when computing the TD target. Otherwise gradients flow through both V(s) and V(s'), causing the critic to push V(s) toward the target AND push V(s') away from the target simultaneously
- The advantage must also be detached before computing policy loss — it's a fixed signal for the actor, not something to differentiate through
- The critic learning rate should be faster than the actor (1.5-3x) so the critic can provide accurate advantage estimates before the policy shifts too far
- Gradient accumulation over a batch of episodes before calling optimizer.step() stabilizes training
- The negative sign in the policy loss converts gradient ascent (maximize expected reward) into gradient descent (what optimizers do)

In [ ]:
import os

import gymnasium as gym
import structlog
import torch
from cart_pole_logger import CartPoleLogger

from samsara_rl.control.actor_critic.value_function_actor_critic import (
    ValueFunctionActorCritic,
)
from samsara_rl.functions.neural_networks.fully_connected import FullyConnected
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

%load_ext tensorboard

In [ ]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/actor_critic_cart_pole"
ALPHAS = [0.05, 0.05 / 2, 0.05 + 0.025]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005] 0.0001, 0.0003, 0.001,
GAMMAS = [0.99, 0.95, 0.90]

In [ ]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single actor-critic configuration.

    Creates value and policy networks with separate optimizers,
    trains for MAX_EPISODES episodes, and logs metrics to TensorBoard.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the policy optimizer. Critic uses alpha * 1.75.
        gamma: Discount factor.
    """
    model_name = "actor_critic"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    value_network = FullyConnected(4, 32, 1)
    policy_network = FullyConnected(4, 32, 2)

    value_optimizer = torch.optim.Adam(value_network.parameters(), lr=alpha * 1.75)

    policy_optimizer = torch.optim.Adam(policy_network.parameters(), lr=alpha)

    agent = ValueFunctionActorCritic(
        env,
        gamma=gamma,
        alpha=alpha,
        batch_size=32,
        value_network=value_network,
        policy_network=policy_network,
        experiment_name=run_name,
        log_dir=run_dir,
        policy_optimizer=policy_optimizer,
        value_optimizer=value_optimizer,
    )
    agent.register(CartPoleLogger())
    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [ ]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [ ]:
%tensorboard --logdir logs/actor_critic_cart_pole

In [ ]:
main()

### Analysis

A well trained value network should output low values for states with high pole angle because they are closer to terminating the episode and higher values for states closer to upright.

A well trained policy network should output a higher probability of LEFT when the cart is falling RIGHT to balance the forces and pull it upright.

This is the episode reward graph of actor critic - note the five phases:

1. Phase 1: 0-500 Episodes, No change
2. Phase 2: 500-750 Episodes, High Slope Growth to 450 Reward
3. Phase 3: 750-1200 Episodes, Collapse to 0
4. Phase 4: 1200-1800 Episodes, Growth to 400
5. Phase 5: 1800+ Steps, Episodes to 450-500


### The Graph Below has Gamma=0.9, Alpha=0.025

![Alt text](reward.png)

### Phase 0 - Random Velocity & Policy Networks

### This is the critic estimate for a centered stationary card. It's randomly initialized and therefore outputs terrible estimates of long term reward. It thinks 0,0 no tilt or angular velocity is actuallly one of the worst positions since its near the local minimia. 

### The first 500 episodes the value function is warming up. The advantage estimate depends on V(S) and V(S') however both estimates are initially innacurate for ALMOST all states

![Alt text](initial_value_estimate.png)

### The only exception is for terminal states. Right before the pole falls the advantage estimate is: 
### R + gamma * V(S') - V(S); V(S') is 0  (definition of terminal state)

### Final Critic Estimate
#### Critic has learned the concave shape: closer to the origin with no tilt and no angular velocity leads to higher long term rewards. Very tilted states with high angular velocity especially to the right has less reward. Notice it's only by a few though because it has learned close to an optimal policy. So even in a very tilted state it can recover and get 6.6 discounted reward

![Alt text](final_value.png )

###  So the first signal that the Critic learns is states right before terminal states are bad. This constant signal eventually ripples through states adjacent to states that are adjacent to terminal states and eventually shapes the entire value surface to say states closer to upright are better.

### Why the Collapse mid training
#### The policy and value functions influence each  other - the critic tells the actor to  magnify actions that lead to states with higher estimate of future reward. So eventually the policy begins taking different actions and therefore entering new states that the critic has never seen before.

#### Each of the three graphs below shows the percentage of time spent in high angle (dangerous), low angle (safe), and mid angle states. Initially the pole is moving randomly and spends almost equal time in dangerous high angle states as it does safe low angle states (episodes are short). As it does better and better, the pole spends almost all of its time in upright positions which initially the critic has not seen as much of and therefore outputs terrible estimates that contribute to the collapse. 

#### Eventually it learns to model these new states and the reward picks up again and goes towards 450-500. The policy network needs to move slower than the critic and is given a slower learning rate so the critic isn't trying to rapidly chase a constantly shifting policy

![Alt text](visitation.png )